In [2]:
import pandas as pd
import numpy as np
import os
import shutil
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
breeds = {
        1 : 'Abyssinian',
        2 : 'Bengal',
        3 : 'Birman',
        4 : 'Bombay',
        5 : 'British_Shorthair',
        6 : 'Egyptian_Mau',
        7 : 'Maine_Coon',
        8 : 'Persian',
        9 : 'Ragdoll',
        10 : 'Russian_Blue',
        11 : 'Siamese',
        12 : 'Sphynx'        
    }

train_df = pd.read_csv('dataset/annotations/trainval.txt', header=None, sep=' ')
train_df = train_df.iloc[(train_df.iloc[:,2] == 1).values,[0,3]]
train_df = train_df.rename(columns={0 : 'file_name', 3:'breed'})
train_df['breed_name'] = train_df['breed'].map(breeds)
train_df.drop_duplicates(keep=False,inplace=True)

train_set = train_df.sample(frac=0.8, random_state=42)
val_set = train_df.drop(train_set.index)


test_df = pd.read_csv('dataset/annotations/test.txt', header=None, sep=' ')
test_df = test_df.iloc[(test_df.iloc[:,2] == 1).values,[0,3]]
test_df = test_df.rename(columns={0 : 'file_name', 3:'breed'})
test_df['breed_name'] = test_df['breed'].map(breeds)
test_df.drop_duplicates(keep=False,inplace=True)


           file_name  breed  breed_name
1019  Maine_Coon_128      7  Maine_Coon
3498      Sphynx_179     12      Sphynx
2853  Maine_Coon_175      7  Maine_Coon
286       Bengal_133      2      Bengal
2882  Maine_Coon_214      7  Maine_Coon
         file_name  breed  breed_name
1   Abyssinian_101      1  Abyssinian
8   Abyssinian_108      1  Abyssinian
13  Abyssinian_112      1  Abyssinian
14  Abyssinian_113      1  Abyssinian
20  Abyssinian_119      1  Abyssinian
        file_name  breed  breed_name
0  Abyssinian_201      1  Abyssinian
1  Abyssinian_202      1  Abyssinian
2  Abyssinian_204      1  Abyssinian
3  Abyssinian_205      1  Abyssinian
4  Abyssinian_206      1  Abyssinian


In [4]:
def move_images(source, destination, df):
    for index, row in df.iterrows():
        source_path = source+row['file_name']+'.jpg'
        destination_path = destination+row['breed_name']
        if not os.path.exists(destination_path):
            os.makedirs(destination_path)
        try:
            shutil.copy(source_path,destination_path)
        except Exception as e:
            print('error :',e)
        


move_images('dataset/images/','dataset/train/',train_set)
move_images('dataset/images/','dataset/validation/',val_set)
move_images('dataset/images/','dataset/test/',test_df)

In [15]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Load dataset
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset/validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "dataset/test",
    image_size = IMG_SIZE,
    batch_size=BATCH_SIZE
)

num_classes = len(breeds)

normalization_layer = layers.Rescaling(1./255)

Found 950 files belonging to 12 classes.
Found 238 files belonging to 12 classes.
Found 1183 files belonging to 12 classes.


In [8]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 12)             │         1,548 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,170,508 (42.61 MB)

 Trainable params: 11,170,508 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=200
)

Epoch 1/200
30/30 ━━━━━━━━━━━━━━━━━━━━ 28s 925ms/step - accuracy: 0.0874 - loss: 2.4846 - val_accuracy: 0.0714 - val_loss: 2.4865
Epoch 2/200
30/30 ━━━━━━━━━━━━━━━━━━━━ 25s 832ms/step - accuracy: 0.0874 - loss: 2.4846 - val_accuracy: 0.0714 - val_loss: 2.4866
Epoch 3/200
30/30 ━━━━━━━━━━━━━━━━━━━━ 28s 931ms/step - accuracy: 0.0874 - loss: 2.4846 - val_accuracy: 0.0714 - val_loss: 2.4867
Epoch 4/200
30/30 ━━━━━━━━━━━━━━━━━━━━ 25s 838ms/step - accuracy: 0.0874 - loss: 2.4845 - val_accuracy: 0.0714 - val_loss: 2.4868
Epoch 5/200
30/30 ━━━━━━━━━━━━━━━━━━━━ 24s 802ms/step - accuracy: 0.0874 - loss: 2.4845 - val_accuracy: 0.0714 - val_loss: 2.4869
Epoch 6/200
30/30 ━━━━━━━━━━━━━━━━━━━━ 24s 808ms/step - accuracy: 0.0874 - loss: 2.4845 - val_accuracy: 0.0714 - val_loss: 2.4871
Epoch 7/200
30/30 ━━━━━━━━━━━━━━━━━━━━ 24s 804ms/step - accuracy: 0.0768 - loss: 2.4845 - val_accuracy: 0.0714 - val_loss: 2.4871
Epoch 8/200
30/30 ━━━━━━━━━━━━━━━━━━━━ 24s 806ms/step - accuracy: 0.0863 - loss: 2.4844 - 

In [17]:
loss, acc = model.evaluate(test_ds)
print("Validation accuracy:", acc)

37/37 ━━━━━━━━━━━━━━━━━━━━ 9s 225ms/step - accuracy: 0.0845 - loss: 2.4849
Validation accuracy: 0.08453085273504257
